# 04 — Recurrent models for order (RNN / LSTM)

The bag-of-words model from notebook 03 is blind to word order. Here we (1) build a task where **order is the whole point**, (2) confirm the mean-pool model fails on it, and (3) solve it with an **LSTM** — a model that reads the sequence step by step and carries a memory. Then we do a little **char-level text generation** to see a sequence model produce output. (Ties to Part 4 of the curriculum.)

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Data — "does token A appear before token B?"

Each sequence contains token `A` and token `B` exactly once, at random positions; label = 1 if `A` comes before `B`. Same *multiset* either way, so bag-of-words sees identical inputs for both labels — only **order** distinguishes them.

In [ ]:
V, S, A, B = 12, 16, 3, 8
def make_order(n):
    seqs, ys = [], []
    for _ in range(n):
        s = torch.randint(0, V, (S,)); s[s == A] = 0; s[s == B] = 0
        i, j = np.random.choice(S, 2, replace=False)
        s[i] = A; s[j] = B
        seqs.append(s); ys.append(int(i < j))
    return torch.stack(seqs), torch.tensor(ys)

Xtr, ytr = make_order(4000); Xva, yva = make_order(1000)
Xtr, ytr, Xva, yva = (t.to(device) for t in (Xtr, ytr, Xva, yva))

def fit(model, epochs, lr=1e-2, bs=128):
    model = model.to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr); hist = []
    for _ in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for k in range(0, len(Xtr), bs):
            idx = perm[k:k+bs]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            hist.append((model(Xva).argmax(1) == yva).float().mean().item())
    return hist

## Bag-of-words fails (~chance)

The mean-pool classifier from notebook 03, on this order task:

In [ ]:
class MeanPool(nn.Module):
    def __init__(s, V, D=32):
        super().__init__(); s.emb = nn.Embedding(V, D); s.fc = nn.Linear(D, 2)
    def forward(s, x): return s.fc(s.emb(x).mean(1))

bow_hist = fit(MeanPool(V), epochs=30)
print(f"mean-pool val acc: {bow_hist[-1]:.3f}   <- ~0.5, blind to order")

## An LSTM solves it

`nn.LSTM` reads the sequence one token at a time, maintaining a hidden state (its memory). We take the **final hidden state** as a summary and classify it. Because it processes tokens *in order*, it can record "which of A/B did I see first."

`batch_first=True` makes it expect `(B, S, D)`. `nn.LSTM` returns `output` (all steps) and `(h_n, c_n)` (final hidden/cell state); we use `h_n`.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, V, D=32, H=48):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.lstm = nn.LSTM(D, H, batch_first=True)
        self.fc = nn.Linear(H, 2)
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(self.emb(x))   # out (B,S,H); h_n (1,B,H)
        return self.fc(h_n[-1])                     # final hidden state -> logits

lstm_hist = fit(LSTMClassifier(V), epochs=40)
print(f"LSTM val acc: {lstm_hist[-1]:.3f}   <- solves the order task")

plt.plot(bow_hist, label="mean-pool (bag of words)")
plt.plot(lstm_hist, label="LSTM")
plt.axhline(0.5, ls=":", c="gray"); plt.xlabel("epoch"); plt.ylabel("val accuracy")
plt.title('"A before B?" — order matters'); plt.legend(); plt.show()

The gap is the lesson: **recurrence gives the model access to order**, which pooling discarded. (This is exactly why RNNs/LSTMs mattered for sequence modeling — Part 4.)

## Bonus — char-level generation

A sequence model can also *generate*. Train a tiny char-LSTM to predict the next character on a small repeating corpus, then sample from it.

In [ ]:
text = "hello world. this is a tiny corpus. " * 200
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])

class CharLSTM(nn.Module):
    def __init__(self, V, D=48, H=96):
        super().__init__()
        self.emb = nn.Embedding(V, D); self.lstm = nn.LSTM(D, H, batch_first=True); self.head = nn.Linear(H, V)
    def forward(self, x, state=None):
        out, state = self.lstm(self.emb(x), state)
        return self.head(out), state

clm = CharLSTM(len(chars)).to(device)
opt = torch.optim.AdamW(clm.parameters(), lr=3e-3)
BLK = 32
for step in range(400):
    i = torch.randint(0, len(data) - BLK - 1, (64,))
    x = torch.stack([data[j:j+BLK] for j in i]).to(device)
    y = torch.stack([data[j+1:j+1+BLK] for j in i]).to(device)
    logits, _ = clm(x)
    loss = F.cross_entropy(logits.reshape(-1, len(chars)), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print("final train loss:", round(loss.item(), 3))

In [ ]:
@torch.no_grad()
def generate(model, prompt="hello", n=60):
    model.eval()
    idx = torch.tensor([[stoi[c] for c in prompt]], device=device)
    state = None; logits, state = model(idx, state); out = prompt
    for _ in range(n):
        probs = F.softmax(logits[:, -1], dim=-1)
        nxt = torch.multinomial(probs, 1)
        out += itos[nxt.item()]
        logits, state = model(nxt, state)
    return out

print(generate(clm, "hello", 60))

It reproduces the corpus's structure — a sequence model turning a learned next-token distribution into text (the same idea that scales up to LLMs, minus the scale and the Transformer).

## Takeaways

- **`nn.LSTM`** reads a sequence step by step and carries a hidden state (memory) — so it *sees order*, which pooling can't.
- Use the **final hidden state** to classify a whole sequence; use **per-step outputs** for next-token prediction / generation.
- The generation loop = repeatedly sample from `softmax(logits)` and feed the token back in.

Next: attention — an order-aware model that (unlike the LSTM) processes the whole sequence in parallel (notebook 05).